# 第 5 周 · 周二：窗口函数——聚合但不折叠

> 难度：⭐⭐⭐ | 今日目标：理解窗口函数的核心原理，掌握8种常用窗口函数
> 配对课程：01python_learn W5D2（时间序列操作）
> 前置依赖：周一SQL分析思维、基础SELECT/GROUP BY
> 扩展阅读：02analyst_thinking/week_01/day_01（SQL窗口函数完整知识图谱）
> 面试地位：大厂SQL面试必考，10题中有3-4题涉及窗口函数

---

## 一、核心概念与第一性原理

### 1.1 GROUP BY 的困境——为什么需要窗口函数

SQL是一门集合操作语言。传统的GROUP BY能把多行数据压缩成一行——你可以看到每个组的聚合结果，但失去了行级的上下文信息。

```sql
-- GROUP BY：把1000行压成1行
SELECT user_id, SUM(amount) FROM orders GROUP BY user_id;
-- 你能看到每个用户的总消费，但看不到每笔订单的细节
```

然而业务分析常常需要"行级聚合"——既要看到每一行原始数据，又要看到基于其他行计算的聚合值：
- 每个人的累计消费（既要看到每行订单，又要看到累计值）
- 每人每天的消费排名（既要看到当天数据，又要看到排名）
- 每人的环比变化（既要看到本月和上月数据，又要看到差值）

### 1.2 窗口函数的魔法——聚合不折叠

```sql
SELECT user_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date) AS cumulative_sum
FROM orders;
```

输出：每一笔订单都在，但多了一列"该用户截止到这笔订单的累计消费"。

**窗口函数 = GROUP BY的能力 + 不折叠行。**

它填补了SQL作为集合语言的一个根本性空白——"既要保留每一行，又要参考其他行做计算"。

### 1.3 窗口函数的三大组件

```
函数名(列) OVER (PARTITION BY 分组列 ORDER BY 排序列 窗口帧)
  │        │          │              │            │
  │        要计算的列  定义"和谁比"   定义"按什么顺序比"  定义"比的范围"
  SUM/ROW_NUMBER/RANK/LAG...
```

- **PARTITION BY**：定义"和谁比"。比如按user_id分区，就是"每个人内部自己和自己比"
- **ORDER BY**：定义"按什么顺序比"。比如按order_date排序，就是"按时间顺序比"
- **窗口帧（ROWS/RANGE）**：定义"看多大范围"。比如ROWS BETWEEN 3 PRECEDING AND CURRENT ROW，就是"包括当前行在内的最近4行"

这三个子句缺一不可——没有PARTITION BY就是全体比，没有ORDER BY就是乱序比，没有窗口帧就是全部比。

### 1.4 窗口函数 vs GROUP BY vs 自连接——方案对比

| 场景 | 窗口函数 | GROUP BY | 自连接 |
|------|---------|---------|--------|
| 保留行级数据 | ✅ 保留每行 | ❌ 折叠行 | ✅ 但行数膨胀 |
| 计算聚合值 | ✅ | ✅ | ✅ |
| 代码简洁度 | 最高（一行OVER） | 中等 | 最低（多层嵌套） |
| 性能（大数据） | 好（一次扫描） | 好 | 差（多次扫描） |

**选择策略**：
- 需要行级数据+聚合值 → 窗口函数
- 只需要聚合值 → GROUP BY
- 需要复杂的行级比较（非聚合） → 自连接

### 1.5 窗口函数的执行逻辑

数据库执行窗口函数时，会按以下顺序操作：
1. 先执行FROM/JOIN/WHERE得到中间结果集
2. 按PARTITION BY分组
3. 在每个分组内按ORDER BY排序
4. 按ROWS/RANGE定义窗口范围
5. 对每个窗口执行窗口函数计算

**重点**：ORDER BY只在窗口内生效，不会改变整个查询的最终排序。查询的最终排序由查询尾部的ORDER BY决定。

### 1.6 面试高频考点

**面试官常问**：
- "窗口函数和GROUP BY有什么区别？"
- "什么时候用窗口函数，什么时候用GROUP BY？"
- "ROW_NUMBER、RANK、DENSE_RANK的区别是什么？"
- "LAG和LEAD的用途是什么？"

**回答要点**：
- 窗口函数保留行级数据，GROUP BY折叠数据
- 需要同时展示原始数据和聚合结果时用窗口函数
- ROW_NUMBER强制唯一，RANK跳跃，DENSE_RANK连续
- LAG/LEAD用于时间序列分析，计算环比、同比

> **今天最重要的认知**：窗口函数填补了SQL作为集合语言的空白——"既要保留每一行，又要参考其他行做计算"。这不是语法糖，这是从"会写SQL"到"能用SQL思考"的升级。

---

## 二、知识点详细拆解

### 2.1 ROW_NUMBER / RANK / DENSE_RANK —— 三种排名函数

**为什么重要**：排名是最常见的业务需求之一，三种函数的选择直接影响业务逻辑。

| 函数 | 并列值处理 | 输出示例(分数:95,95,90) | 适用场景 |
|------|-----------|-----------------------|----------|
| ROW_NUMBER | 强制唯一 | 1, 2, 3 | 严格取前N个（控制预算） |
| RANK | 跳跃排名 | 1, 1, 3 | 并列算同名次（比赛排名） |
| DENSE_RANK | 连续排名 | 1, 1, 2 | 并列但不跳跃（等级划分） |

**选择指南**：
- 运营要发优惠券，只有3张 → ROW_NUMBER（严格3个）
- 销售排行榜展示 → RANK（并列第一名两人）
- 绩效等级（前20%A，20-50%B） → DENSE_RANK（避免跳跃导致断层）

### 2.2 LAG / LEAD —— 向前/向后取行

**为什么重要**：时间序列分析的基石，几乎所有环比、同比、时间间隔计算都依赖这两个函数。

```sql
LAG(column, offset, default) OVER (PARTITION BY ... ORDER BY ...)
LEAD(column, offset, default) OVER (PARTITION BY ... ORDER BY ...)
```

**典型用法**：
- 计算环比：`amount / LAG(amount) OVER (ORDER BY date) - 1`
- 计算用户两次购买间隔：`DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY user_id ORDER BY order_date))`
- 查看下一个订单金额：`LEAD(amount) OVER (PARTITION BY user_id ORDER BY order_date)`
- 同比计算：`LAG(amount, 12) OVER (PARTITION BY category ORDER BY month)`

### 2.3 SUM/AVG/COUNT + OVER —— 累计值计算

**为什么重要**：累计GMV、累计用户数、滚动平均等指标是运营日报的核心内容。

```sql
-- 累计和
SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date)

-- 3期移动平均
AVG(amount) OVER (PARTITION BY user_id ORDER BY order_date
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)

-- 每人的订单总数
COUNT(*) OVER (PARTITION BY user_id)

-- 每人消费占其个人总消费的比例
amount / SUM(amount) OVER (PARTITION BY user_id)
```

**窗口帧的关键**：
- 默认（有ORDER BY）：`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`——从第一行到当前行
- 物理行窗口：`ROWS BETWEEN 3 PRECEDING AND CURRENT ROW`——最近4行
- 逻辑值窗口：`RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW`——最近7天

### 2.4 FIRST_VALUE / LAST_VALUE —— 窗口首尾值

**为什么重要**：取每组的最早/最晚状态，比如用户第一次购买的商品、最后登录时间等。

```sql
-- 首次购买商品
FIRST_VALUE(product_id) OVER (PARTITION BY user_id ORDER BY order_date)

-- 最新状态（注意：需要指定窗口帧到最后一行）
LAST_VALUE(status) OVER (
    PARTITION BY user_id ORDER BY update_time
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
)
```

### 2.5 ROWS vs RANGE —— 物理窗口 vs 逻辑窗口

**为什么重要**：错误的窗口帧会导致错误的计算结果。

| 类型 | 定义方式 | 适用场景 |
|------|---------|----------|
| ROWS | 按物理行数 | 移动平均（最近N行），不管日期是否连续 |
| RANGE | 按值范围 | 最近N天的数据（如果某天没数据自动跳过） |

```sql
-- ROWS：最近7行（物理行）
AVG(amount) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)

-- RANGE：最近7天（逻辑范围）
AVG(amount) OVER (ORDER BY date RANGE BETWEEN INTERVAL '6' DAY PRECEDING AND CURRENT ROW)
```

**如果7天中只有5天有数据，ROWS取7行（可能跨更多天），RANGE只取5行（严格7天内）**。

### 2.6 PARTITION BY 多列 —— 多维分组

**为什么重要**：实际业务中经常需要按多个维度分组。

```sql
-- 按城市+品类分组，计算每个组内的累计销售额
SELECT city, category, date, sales,
       SUM(sales) OVER (PARTITION BY city, category ORDER BY date) AS cum_sales
FROM daily_sales;
```

### 2.7 ORDER BY 在窗口内 vs 查询尾部

**为什么重要**：很多人混淆这两个ORDER BY的作用域。

```sql
SELECT user_id, order_date, amount,
       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_date) AS rn
FROM orders
ORDER BY amount DESC;  -- 整个查询的排序，和窗口内的ORDER BY无关
```

窗口内的ORDER BY只影响窗口函数的计算，查询尾部的ORDER BY才是最终输出的排序。

### 2.8 窗口函数性能注意事项

**为什么重要**：大数据量下窗口函数可能成为瓶颈。

- 避免不必要的ORDER BY（不需要顺序时去掉ORDER BY可以提升性能）
- PARTITION BY的列如果倾斜严重（某个值占了90%数据），该分区会非常慢
- 窗口函数不能嵌套使用，但可以放在不同CTE层
- Hive中窗口函数会触发额外的Shuffle阶段，数据倾斜时会特别慢

---

## 三、动手练习

使用Python的sqlite3模拟窗口函数（SQLite 3.35+支持完整窗口函数）。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 创建模拟订单表（200行，30个用户，4个品类）
n = 200
df = pd.DataFrame({
    'user_id': np.random.choice(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'], n),
    'order_date': np.random.choice(pd.date_range('2026-01-01', '2026-01-31'), n),
    'amount': np.round(np.random.uniform(50, 500, n), 2),
    'city': np.random.choice(['北京', '上海', '广州', '深圳'], n),
    'category': np.random.choice(['服装', '数码', '食品', '美妆'], n)
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

print(f"数据: {n} 行, {df['user_id'].nunique()} 个用户")
print(f"日期范围: {df['order_date'].min()} ~ {df['order_date'].max()}")
print()

### 练习1：ROW_NUMBER vs RANK vs DENSE_RANK

In [ ]:
print("=" * 60)
print("练习1：三种排名函数的并列值处理差异")
print("=" * 60)

q1 = """SELECT user_id, amount,
  ROW_NUMBER() OVER(ORDER BY amount DESC) AS row_num,
  RANK() OVER(ORDER BY amount DESC) AS rank_num,
  DENSE_RANK() OVER(ORDER BY amount DESC) AS dense_rank_num
FROM orders
ORDER BY amount DESC
LIMIT 15"""
result1 = pd.read_sql(q1, conn)
print(result1)

print()
print("观察：")
print("  - ROW_NUMBER: 每行都不同（强制唯一）")
print("  - RANK: 相同amount行号相同，但会跳号")
print("  - DENSE_RANK: 相同amount行号相同，但不跳号")
print()
print("如果amount 前3行分别是500, 500, 499：")
print("  ROW_NUMBER: 1, 2, 3")
print("  RANK:        1, 1, 3")
print("  DENSE_RANK:  1, 1, 2")

### 练习2：每人累计消费和消费占比

In [ ]:
print("=" * 60)
print("练习2：每人累计消费 + 消费占比")
print("=" * 60)

q2 = """SELECT user_id, order_date, amount,
  SUM(amount) OVER(PARTITION BY user_id ORDER BY order_date) AS cumulative,
  SUM(amount) OVER(PARTITION BY user_id) AS user_total,
  ROUND(amount * 100.0 / SUM(amount) OVER(PARTITION BY user_id), 1) AS pct_of_user
FROM orders
ORDER BY user_id, order_date
LIMIT 15"""
result2 = pd.read_sql(q2, conn)
print(result2)
print()
print("  cumulative列：截止到本行的累计消费")
print("  user_total列：该用户的总消费（每行都一样）")
print("  pct_of_user列：本笔订单占该用户总消费的百分比")

### 练习3：每人环比变化（LAG）

In [ ]:
print("=" * 60)
print("练习3：每人订单金额环比变化")
print("=" * 60)

q3 = """SELECT user_id, order_date, amount,
  LAG(amount) OVER(PARTITION BY user_id ORDER BY order_date) AS prev_amount,
  ROUND(amount - LAG(amount) OVER(PARTITION BY user_id ORDER BY order_date), 2) AS diff,
  CASE
    WHEN LAG(amount) OVER(PARTITION BY user_id ORDER BY order_date) IS NULL THEN '首单'
    WHEN amount > LAG(amount) OVER(PARTITION BY user_id ORDER BY order_date) THEN '↑增长'
    WHEN amount < LAG(amount) OVER(PARTITION BY user_id ORDER BY order_date) THEN '↓下降'
    ELSE '→持平'
  END AS trend
FROM orders
ORDER BY user_id, order_date
LIMIT 15"""
result3 = pd.read_sql(q3, conn)
print(result3)
print()
print("  prev_amount: 上一笔订单的金额")
print("  diff: 比上一笔多了多少")
print("  trend: 变化趋势（首单/增长/下降/持平）")

### 练习4：进阶练习——每个城市消费Top3、3天移动平均

In [ ]:
print("=" * 60)
print("练习4A：每个城市消费Top3用户")
print("=" * 60)

q4a = """SELECT city, user_id, total_amount, city_rank
FROM (
  SELECT city, user_id, SUM(amount) AS total_amount,
    ROW_NUMBER() OVER(PARTITION BY city ORDER BY SUM(amount) DESC) AS city_rank
  FROM orders
  GROUP BY city, user_id
)
WHERE city_rank <= 3
ORDER BY city, city_rank"""
result4a = pd.read_sql(q4a, conn)
print(result4a)

print()
print("=" * 60)
print("练习4B：每日GMV的3天移动平均")
print("=" * 60)

q4b = """SELECT order_date,
  SUM(amount) AS daily_amount,
  ROUND(AVG(SUM(amount)) OVER (
    ORDER BY order_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ), 2) AS moving_avg_3d
FROM orders
GROUP BY order_date
ORDER BY order_date
LIMIT 10"""
result4b = pd.read_sql(q4b, conn)
print(result4b)
print()
print("  daily_amount: 当天GMV")
print("  moving_avg_3d: 当天+前两天的移动平均（前两行可能不足3天）")

### 练习5：挑战练习——找出每人首次超过个人平均值的订单

In [ ]:
print("=" * 60)
print("练习5：找出每人首次超过个人平均值的订单")
print("=" * 60)

q5 = """WITH user_avg AS (
    SELECT user_id, AVG(amount) AS avg_amount
    FROM orders
    GROUP BY user_id
),
ordered AS (
    SELECT o.user_id, o.order_date, o.amount, ua.avg_amount,
        ROW_NUMBER() OVER(PARTITION BY o.user_id ORDER BY o.order_date) AS seq
    FROM orders o
    JOIN user_avg ua ON o.user_id = ua.user_id
),
first_over AS (
    SELECT user_id, order_date, amount, avg_amount, seq,
        MIN(CASE WHEN amount > avg_amount THEN seq END)
            OVER(PARTITION BY user_id) AS first_over_seq
    FROM ordered
)
SELECT user_id, order_date, amount,
    ROUND(avg_amount, 2) AS user_avg,
    ROUND((amount - avg_amount) / avg_amount * 100, 1) AS over_pct
FROM first_over
WHERE seq = first_over_seq
ORDER BY user_id"""
result5 = pd.read_sql(q5, conn)
print(result5)
print()
print("  含义：user_id=A的用户，在第X天首次消费超过其个人平均值")
print("  这个查询综合运用了：CTE + JOIN + ROW_NUMBER + 条件窗口函数")

---

## 四、业务场景应用

### 场景1：运营要发"每个城市消费金额Top3用户"优惠券

**问题**：用ROW_NUMBER还是RANK？为什么这个选择影响预算？

**分析**：
- ROW_NUMBER：不管有没有并列，每个城市严格取3人。预算固定。
- RANK：如果有并列第3名，会取多个（比如并列第2有2人，那总共有4人上榜）。预算超支。
- DENSE_RANK：并列第2名有2人，第3名仍然存在。

**决策**：预算有限必须控制人数 → ROW_NUMBER。追求公平并列都发 → RANK。

**SQL（ROW_NUMBER严格取3人）**：
```sql
SELECT * FROM (
  SELECT city, user_id, SUM(amount) AS total,
    ROW_NUMBER() OVER(PARTITION BY city ORDER BY SUM(amount) DESC) AS rn
  FROM orders GROUP BY city, user_id
) WHERE rn <= 3;
```

### 场景2：财务要做月度GMV滚动3月均值

**问题**：为什么必须用ROWS而不能用RANGE？

**分析**：
- RANGE：假设每月有不同天数，RANGE会按日期值范围计算，可能包含不同月份的数据
- ROWS：按物理行数计算，严格取最近3行（3个月）

**决策**：必须用ROWS才能保证严格的3个月窗口。

### 场景3：产品要分析用户购买间隔分布

**问题**：如何计算用户两次购买之间的时间间隔？

**SQL**：
```sql
SELECT user_id, order_date,
       DATEDIFF(order_date, LAG(order_date) OVER (
           PARTITION BY user_id ORDER BY order_date
       )) AS days_since_last_purchase
FROM orders;
```

**洞察**：如果days_since_last_purchase的中位数在增长，说明复购周期拉长，需要召回活动。

### 场景4：实时累计看板——CEO要看本月GMV累计趋势

**SQL**：
```sql
SELECT order_date,
       SUM(amount) AS daily_gmv,
       SUM(SUM(amount)) OVER (ORDER BY order_date) AS cumulative_gmv
FROM orders
WHERE order_date >= '2026-01-01'
GROUP BY order_date
ORDER BY order_date;
```

这个看板每天CEO都会看——累计曲线是否平滑增长、有没有明显的"断崖"。

---

## 五、分级检验清单

### 初级检验（必须能独立完成）
- [ ] 能写出含PARTITION BY + ORDER BY的ROW_NUMBER
- [ ] 能写出LAG实现环比
- [ ] 能写出SUM() OVER()实现累计
- [ ] 能区分ROW_NUMBER/RANK/DENSE_RANK的输出差异

**答案提示**：
- ROW_NUMBER强制唯一，RANK跳跃，DENSE_RANK连续
- LAG(amount) OVER (PARTITION BY user_id ORDER BY order_date)
- SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date)

### 中级检验（能解释给同事听）
- [ ] 能用白话解释ROW_NUMBER和RANK在并列值时的输出差异
- [ ] 能说出窗口函数和GROUP BY的核心区别
- [ ] 能解释ROWS和RANGE在窗口帧中的差异
- [ ] 能判断什么时候用窗口函数 vs GROUP BY vs 自连接

**答案提示**：
- ROW_NUMBER给每行一个唯一序号，即使值相同；RANK跳过并列名次
- 窗口函数保留行级数据，GROUP BY折叠行
- ROWS按物理行数，RANGE按值范围
- 需要行级+聚合→窗口函数，只要聚合→GROUP BY，复杂行级比较→自连接

### 高级检验（能举一反三）
- [ ] 能解决"找出每人首次超过平均值"这类需要组合窗口函数的问题
- [ ] 能估算窗口函数在大数据量下的性能影响
- [ ] 能设计多层嵌套窗口函数解决复杂问题
- [ ] 面对"行级上下文计算"需求本能地想到窗口函数

**答案提示**：
- 组合使用窗口函数和条件聚合
- 大数据量下减少PARTITION BY的基数、避免不必要的ORDER BY
- 多层窗口函数放在不同CTE层
- 一旦需要"在不折叠行的前提下参考其他行"，就是窗口函数的场景

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 昨日** | SQL分析思维 | 窗口函数是"分析型SQL"最重要的工具 |
| **← 基础** | GROUP BY | 窗口函数是GROUP BY的延伸——聚合但不折叠 |
| **→ 明日** | 连续登录与留存 | 用ROW_NUMBER做日期编号，实现gap-and-island |
| **→ 第6周** | 同期群分析/漏斗 | 这些分析的SQL实现都依赖窗口函数 |
| **业务连接** | 所有"排名""环比""累计"报表 | 运营日报、财务月报、用户分析 |
| **扩展阅读** | 02analyst_thinking W1D1 | 窗口函数完整知识图谱+面试题合集 |

### 横向连接：窗口函数与Pandas的对等操作

| SQL窗口函数 | Pandas对等操作 |
|------------|---------------|
| ROW_NUMBER() OVER(...) | df.groupby('col').cumcount() + 1 |
| RANK() OVER(...) | df.groupby('col')['val'].rank(method='min') |
| LAG(amount) OVER(...) | df.groupby('col')['amount'].shift(1) |
| SUM() OVER(ORDER BY ...) | df['amount'].cumsum() |
| AVG() OVER(ROWS ...) | df['amount'].rolling(7).mean() |

---

## 七、当日小结

> **今天最重要的一个认知**：窗口函数 = GROUP BY的能力 + 不折叠行。它的设计哲学是"给每一行提供它所在群体的上下文"——让你在保留明细的同时获得汇总视角。8种常用函数：ROW_NUMBER/RANK/DENSE_RANK做排名，LAG/LEAD做前后对比，SUM/AVG/COUNT做累计计算。记住一句话：窗口函数让你在保留行级数据的同时进行聚合计算。

**三条金句带回家**：
1. 窗口函数的本质是"聚合但不折叠"——这是它和GROUP BY的核心区别
2. 排名三兄弟中，ROW_NUMBER严格唯一，RANK跳跃，DENSE_RANK连续
3. 窗口帧的ROWS和RANGE要分清——ROWS按物理行数，RANGE按值范围

### 课后拓展

1. **练习**：用窗口函数实现"找出每个用户购买的前3个品类"（提示：ROW_NUMBER PARTITION BY user_id ORDER BY amount DESC）

2. **思考**：如果订单表有10亿行，窗口函数在Hive上会有什么性能问题？（提示：数据倾斜）

3. **预习**：明天要学的连续登录问题会用到今天的ROW_NUMBER+日期减法

4. **面试准备**：背熟以下问题——"窗口函数和GROUP BY的区别？""ROW_NUMBER/RANK/DENSE_RANK的区别？""LAG函数的用途？"

---

*03analyst_thinking · Week 5 Day 2 · 2026-05-21*